<a href="https://colab.research.google.com/github/AnSeokHyeon1212/BIG_DATA/blob/main/week05_5812911_%EC%95%88%EC%84%9D%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [3]:

# 사용자 정의 Dataset 클래스
class MyDataset(Dataset):

    # 데이터 30개와 특성 4개를 생성
    def __init__(self):
        self.x = torch.randn(30, 4)
        self.y = self.x.sum(dim=1, keepdim=True)

    # 전체 데이터의 개수를 반환
    def __len__(self):
        return len(self.x)

    # 인덱스에 해당하는 입력과 정답을 반환
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# Dataset 생성
dataset = MyDataset()

print("전체 데이터 개수:", len(dataset))
print("입력 데이터 shape:", dataset.x.shape)
print("정답 데이터 shape:", dataset.y.shape)

print("첫 번째 입력:", dataset[0][0])
print("첫 번째 정답:", dataset[0][1])



전체 데이터 개수: 30
입력 데이터 shape: torch.Size([30, 4])
정답 데이터 shape: torch.Size([30, 1])
첫 번째 입력: tensor([-1.1258, -1.1524, -0.2506, -0.4339])
첫 번째 정답: tensor([-2.9627])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [4]:

# Batch Size를 7로 설정
dataloader = DataLoader(dataset, batch_size=7, shuffle=False)

# 전체 Batch 개수
print("전체 Batch 개수:", len(dataloader))

# 각 Batch의 shape 출력
for i, (x, y) in enumerate(dataloader):
    print(f"Batch {i+1}")
    print("입력 shape:", x.shape)
    print("정답 shape:", y.shape)

# 마지막 Batch의 크기가 다른 이유
print("전체 데이터 30개를 7개씩 나누면")
print("7개씩 4묶음과 남은 2개로 총 5개의 Batch가 생성됩니다.")



전체 Batch 개수: 5
Batch 1
입력 shape: torch.Size([7, 4])
정답 shape: torch.Size([7, 1])
Batch 2
입력 shape: torch.Size([7, 4])
정답 shape: torch.Size([7, 1])
Batch 3
입력 shape: torch.Size([7, 4])
정답 shape: torch.Size([7, 1])
Batch 4
입력 shape: torch.Size([7, 4])
정답 shape: torch.Size([7, 1])
Batch 5
입력 shape: torch.Size([2, 4])
정답 shape: torch.Size([2, 1])
전체 데이터 30개를 7개씩 나누면
7개씩 4묶음과 남은 2개로 총 5개의 Batch가 생성됩니다.


## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


**답:** Batch Shape은 `(batch_size, 특성 수)` 형태로 나타납니다.

첫 번째 숫자인 `batch_size`는 한 번에 처리하는 데이터의 개수를 의미하며, 두 번째 숫자인 특성 수는 각 데이터가 가지고 있는 입력값의 개수를 의미합니다.

이번 과제에서는 Batch Size를 7로 설정하고 입력 특성이 4개이므로 입력 Shape은 `(7, 4)`가 됩니다. 정답은 데이터마다 1개씩 존재하므로 정답 Shape은 `(7, 1)`입니다.

전체 데이터가 30개이기 때문에 마지막 Batch에는 2개가 남아 입력 Shape은 `(2, 4)`, 정답 Shape은 `(2, 1)`로 출력됩니다.


## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [5]:

# 1. 선형 모델 생성 (입력 4개, 출력 1개)
model = nn.Linear(4, 1)

# 손실함수와 최적화 방법 설정
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 2. 학습 반복
for epoch in range(30):
    total_loss = 0

    for x, y in dataloader:

        # 1단계: 예측
        pred = model(x)

        # 2단계: 손실 계산
        loss = criterion(pred, y)

        # 3단계: 기울기 초기화
        optimizer.zero_grad()

        # 4단계: 역전파
        loss.backward()

        # 5단계: 가중치 수정
        optimizer.step()

        # Batch별 손실 합산
        total_loss += loss.item() * len(x)

    # Epoch의 평균 손실 계산
    avg_loss = total_loss / len(dataset)

    print(f"Epoch {epoch+1}: 평균 손실 = {avg_loss:.6f}")



Epoch 1: 평균 손실 = 1.325773
Epoch 2: 평균 손실 = 0.181307
Epoch 3: 평균 손실 = 0.026368
Epoch 4: 평균 손실 = 0.003895
Epoch 5: 평균 손실 = 0.000590
Epoch 6: 평균 손실 = 0.000094
Epoch 7: 평균 손실 = 0.000016
Epoch 8: 평균 손실 = 0.000003
Epoch 9: 평균 손실 = 0.000001
Epoch 10: 평균 손실 = 0.000000
Epoch 11: 평균 손실 = 0.000000
Epoch 12: 평균 손실 = 0.000000
Epoch 13: 평균 손실 = 0.000000
Epoch 14: 평균 손실 = 0.000000
Epoch 15: 평균 손실 = 0.000000
Epoch 16: 평균 손실 = 0.000000
Epoch 17: 평균 손실 = 0.000000
Epoch 18: 평균 손실 = 0.000000
Epoch 19: 평균 손실 = 0.000000
Epoch 20: 평균 손실 = 0.000000
Epoch 21: 평균 손실 = 0.000000
Epoch 22: 평균 손실 = 0.000000
Epoch 23: 평균 손실 = 0.000000
Epoch 24: 평균 손실 = 0.000000
Epoch 25: 평균 손실 = 0.000000
Epoch 26: 평균 손실 = 0.000000
Epoch 27: 평균 손실 = 0.000000
Epoch 28: 평균 손실 = 0.000000
Epoch 29: 평균 손실 = 0.000000
Epoch 30: 평균 손실 = 0.000000
